# 08 — Locked Event Probabilities

This notebook converts each locked calibrated temperature
distribution into probabilities over the certified eleven-event
partition.

The construction does not use realised outcomes or market prices.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

def locate_repository(start: Path) -> Path:
    current = start.resolve()

    for candidate in (current, *current.parents):
        if (
            candidate
            / "config/"
            "event_probability_construction_spec.yaml"
        ).exists():
            return candidate

    raise FileNotFoundError("Repository root not found.")

ROOT = locate_repository(Path.cwd())

completed = subprocess.run(
    [
        sys.executable,
        str(
            ROOT
            / "tools/"
            "construct_locked_event_probabilities.py"
        ),
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(completed.stdout)

if completed.returncode != 0:
    print(completed.stderr)
    raise RuntimeError(
        "Locked event probability construction failed."
    )


 LOCKED EVENT PROBABILITY CONSTRUCTION COMPLETE

Selected event-book source: data/processed/certified_event_books.csv
Selected model: pooled_empirical_residual
Selected family: empirical_residual
Locked dispersion scale: 1.25

Prediction rows: 159
Prediction dates: 40
Events per prediction: 11
Probability rows: 1749
Quantile particles: 99
Probability resolution: 0.010101010101010102

Probability books sum to one: True
All particles assigned exactly once: True
Zero probabilities retained: True
Zero-probability event share: 0.39908519153802174
Mean occupied events per book: 6.610062893081761
Mean probability entropy: 1.5888737168281544

Probability regularisation applied: False
Realised outcomes accessed: False
Categorical scores calculated: False
Market prices accessed: False
Trading returns calculated: False

Next stage: select uniform probability mixing using development OOF predictions only, then apply the locked mixing weight to holdout and external probabilities.



## Quantile-particle construction

For settlement date \(d\), decision rule \(r\), and event \(A_{d,j}\),
define

\[
\widehat p_{d,r,j}
=
\frac{1}{99}
\sum_{m=1}^{99}
\mathbf 1
\left\{
\widehat Q_{d,r}\!\left(\frac{m}{100}\right)
\in A_{d,j}
\right\}.
\]

The 99 predictive quantiles are treated as equally weighted
deterministic quadrature particles. They are not interpreted as
independent Monte Carlo draws.

In [2]:
probabilities = pd.read_csv(
    ROOT
    / "outputs/diagnostics/"
    "08_locked_event_probability_panel.csv"
)

summary = pd.read_csv(
    ROOT
    / "outputs/final_tables/"
    "08_event_probability_book_summary.csv"
)

print(
    "Probability rows:",
    len(probabilities),
)

print(
    "Prediction books:",
    probabilities["row_id"].nunique(),
)

print(
    "Events per book:",
    probabilities.groupby("row_id").size().min(),
)

print(
    "Maximum probability-sum error:",
    (
        summary["probability_sum"]
        - 1.0
    ).abs().max(),
)

Probability rows: 1749
Prediction books: 159
Events per book: 11
Maximum probability-sum error: 0.0


## Event boundaries

Every event is left closed and right open. Thus a value equal to an
integer boundary enters the interval beginning at that boundary.

The lowest event is unbounded below and the highest event is
unbounded above. The eleven events therefore form a complete
partition of the real line.

In [3]:
example_row_id = probabilities["row_id"].iloc[0]

example = probabilities.loc[
    probabilities["row_id"].eq(example_row_id),
    [
        "event_order",
        "event_label",
        "quantile_particle_count",
        "raw_event_probability",
    ],
]

print(example.to_string(index=False))

 event_order     event_label  quantile_particle_count  raw_event_probability
           1        T < 24°C                        0               0.000000
           2 24°C ≤ T < 25°C                        0               0.000000
           3 25°C ≤ T < 26°C                        0               0.000000
           4 26°C ≤ T < 27°C                        0               0.000000
           5 27°C ≤ T < 28°C                        0               0.000000
           6 28°C ≤ T < 29°C                        4               0.040404
           7 29°C ≤ T < 30°C                        9               0.090909
           8 30°C ≤ T < 31°C                       15               0.151515
           9 31°C ≤ T < 32°C                       32               0.323232
          10 32°C ≤ T < 33°C                       22               0.222222
          11        T ≥ 33°C                       17               0.171717


## Probability resolution and zero values

Since 99 particles are used, every raw event probability is a multiple
of \(1/99\).

Events containing no particle retain probability zero at this stage.
No clipping, additive constant or uniform mixing is introduced
retrospectively.

In [4]:
manifest = json.loads(
    (
        ROOT
        / "data/manifests/"
        "08_event_probability_manifest.json"
    ).read_text(encoding="utf-8")
)

assert manifest["probability_construction_locked"] is True
assert manifest["probability_regularisation_applied"] is False
assert manifest["realised_outcomes_accessed"] is False
assert manifest["categorical_scores_calculated"] is False
assert manifest["market_prices_accessed"] is False
assert manifest["trading_returns_calculated"] is False

print(
    "Event probability construction locked:",
    True,
)

Event probability construction locked: True


## Evidential boundary

This notebook constructs probability vectors but does not score them.

Uniform probability mixing, categorical evaluation, market comparison
and trading analysis are separate later stages.